<a href="https://colab.research.google.com/github/Prashkov1ch/python-ai-Prashkovich-Anna/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_Lab04_Economists_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 4 (экономисты): Pandas — Series, DataFrame, Index, пропуски, MultiIndex

**Ограничения:** упор на `pandas`, частично `numpy`.

**Цель:** отработать:
- объекты Series/DataFrame/Index,
- индексацию и выборку,
- операции над данными,
- обработку пропусков,
- MultiIndex.

In [1]:
import re
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from google.colab import drive

# Монтируем диск (с защитой от повторного монтирования)
try:
    drive.mount('/content/drive')
except:
    pass

PATH = '/content/drive/MyDrive/цифровая кафедра/'  # Проверь, что имя папки верное!
# Убираем префикс "Копия " из имен файлов, так как в Lab 04 файлы называются без него
FILES = ['Копия 1 Атомэнергопром.xlsx', 'Копия 2 Аэрофлот.xlsx', 'Копия 3 Газпром_петрозаводск.xlsx',
         'Копия 4 Лукойл.xlsx', 'Копия 5 Роснефть.xlsx', 'Копия 6 Самолет.xlsx',
         'Копия 7 Славмо.xlsx', 'Копия 8 Строительная_компания_Век.xlsx',
         'Копия 9 ТГК_1.xlsx', 'Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx']

# --- ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ОЧИСТКИ ЧИСЕЛ ---
def clean_number(value):
    """
    Превращает значение из Excel в число float.
    Удаляет пробелы из строк вида '1 000 000'.
    Если значение пустое — возвращает np.nan.
    """
    if value is None:
        return np.nan
    # Превращаем в строку и убираем все пробелы
    s = str(value).replace(' ', '')
    try:
        return float(s)
    except:
        return np.nan

def org_info(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Сведения об организации"]
    return {
        "file": xlsx_name,
        "company": str(ws.cell(6, 13).value),
        "inn": str(ws.cell(10, 13).value),
        "okved": str(ws.cell(15, 13).value),
        "address": str(ws.cell(16, 13).value),
    }

def parse_financial(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Отчет о финансовых результатах"]
    res = {}
    for r in range(6, 200):
        code = ws.cell(r, 16).value
        if code is None:
            continue
        code = str(code).strip()
        if not re.fullmatch(r"\d{4}", code):
            continue
        v23 = ws.cell(r, 21).value
        v22 = ws.cell(r, 27).value
        # ИСПРАВЛЕНИЕ: используем clean_number для очистки данных
        res[code] = {2022: clean_number(v22), 2023: clean_number(v23)}
    return res

def parse_balance(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Бухгалтерский баланс"]
    res = {}
    for r in range(6, 400):
        code = ws.cell(r, 14).value
        if code is None:
            continue
        code = str(code).strip()
        if not re.fullmatch(r"\d{4}", code):
            continue
        v23 = ws.cell(r, 17).value
        v22 = ws.cell(r, 23).value
        v21 = ws.cell(r, 30).value
        # ИСПРАВЛЕНИЕ: используем clean_number для очистки данных
        res[code] = {2021: clean_number(v21), 2022: clean_number(v22), 2023: clean_number(v23)}
    return res

rows = []
for fn in FILES:
    org = org_info(fn)
    fin = parse_financial(fn)
    bal = parse_balance(fn)

    for year in [2022, 2023]:
        rows.append({
            "company": org["company"],
            "file": org["file"],
            "inn": org["inn"],
            "okved": org["okved"],
            "address": org["address"],
            "year": year,
            "revenue_2110": fin.get("2110", {}).get(year, np.nan),
            "net_profit_2400": fin.get("2400", {}).get(year, np.nan),
            "assets_1600": bal.get("1600", {}).get(year, np.nan),
            "equity_1300": bal.get("1300", {}).get(year, np.nan),
        })

df = pd.DataFrame(rows)
print("✅ Данные успешно загружены и очищены!")
print(f"Shape: {df.shape}")
print(f"Компаний: {df['company'].nunique()}")
print(f"Годы: {sorted(df['year'].unique())}")
df.head()

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'Копия 1 Атомэнергопром.xlsx'

### Задание 1
Создайте `Series` выручки за 2023 год, где индекс — `company`, значения — `revenue_2110`.  
Покажите `index`, `dtype` и первые 5 элементов.

In [ ]:
# Ваш код здесь

### Задание 2
Проверьте, какие компании имеют пропуски (NaN) по `net_profit_2400` в 2023.  
Сделайте булеву маску и выведите список названий компаний.

In [ ]:
# Ваш код здесь

### Задание 3
Выберите из DataFrame только столбцы `company, year, revenue_2110, net_profit_2400`.  
Покажите первые 7 строк.

In [ ]:
# Ваш код здесь

### Задание 4
Создайте новый столбец `margin` = net_profit_2400 / revenue_2110.  
Обработайте деление на 0 и NaN так, чтобы получались NaN. Покажите 10 строк.

In [ ]:
# Ваш код здесь

### Задание 5
Отсортируйте данные за 2023 по `revenue_2110` по убыванию.  
Покажите топ-5 строк (company, revenue_2110).

In [ ]:
# Ваш код здесь

### Задание 6
Используя `Index`, получите пересечение множеств компаний, у которых:  
1) выручка 2023 не NaN; 2) активы 2023 не NaN.  
Выведите список компаний в пересечении.

In [ ]:
# Ваш код здесь

### Задание 7
Установите индекс DataFrame как `company` и выберите данные одной компании через `.loc`.  
Возьмите первую компанию из `df['company'].unique()`.

In [ ]:
# Ваш код здесь

### Задание 8
Сделайте MultiIndex по `(company, year)` и отсортируйте индекс.  
Покажите 6 первых строк.

In [ ]:
# Ваш код здесь

### Задание 9
Из MultiIndex-таблицы получите срез по году 2023 с помощью `xs`.  
Покажите 5 строк.

In [ ]:
# Ваш код здесь

### Задание 10
Заполните пропуски в `margin` медианой по соответствующему году.  
Сделайте это через `groupby('year').transform(...)`. Покажите 10 строк с `margin` и `margin_filled`.

In [ ]:
# Ваш код здесь

### Задание 11
Добавьте столбец `okved_section` — первые 2 цифры ОКВЭД (строка), используя `.str.extract` (regex).  
Покажите 5 строк (company, okved, okved_section).

In [ ]:
# Ваш код здесь

### Задание 12
Создайте логический столбец `profit_pos_2023` (True/False) для строк 2023: прибыль > 0.  
Для строк других лет поставьте NaN. Покажите 10 строк.

In [ ]:
# Ваш код здесь